In [11]:
!pip install instructor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.4/226.4 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 141.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 140.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 44.4 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.12.0
    Uninstalling jiter-0.12.0:
      Successfully uninstalled jiter-0.12.0


In [1]:
!cd /content
!rm -rf macro_financial_forecasting

In [1]:
!git clone https://github.com/chuanbinp/macro_financial_forecasting.git

Cloning into 'macro_financial_forecasting'...
remote: Enumerating objects: 1499, done.
remote: Counting objects: 100% (531/531), done.
remote: Compressing objects: 100% (203/203), done.
remote: Total 1499 (delta 376), reused 344 (delta 328), pack-reused 968 (from 2)
Receiving objects: 100% (1499/1499), 19.42 MiB | 19.77 MiB/s, done.
Resolving deltas: 100% (864/864), done.


In [2]:
%cd macro_financial_forecasting/applications/macro_financial_forecasting/src

/content/macro_financial_forecasting/applications/macro_financial_forecasting/src


In [3]:
from config import Config
from train_data_loader import TrainDataLoader
# from agentics_transducer import AgenticTransducer
from data_model.bloomberg_news_entry import BloombergNewsEntry
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
config = Config("../config.env")

train_data_loader = TrainDataLoader(config)

In [9]:
print("Starting loading pipeline ...")
print(f"Config: {config}")

train_ds = train_data_loader.load()
print("Loading pipeline completed.")

Starting loading pipeline ...
Config: Config(
  gemini_api_key: !secret!
  openai_api_key: !secret!
  llm_model: openai/gpt-5-nano-2025-08-07
  industries: ['Information Technology', 'Health Care', 'Financials', 'Consumer Discretionary', 'Communication Services', 'Industrials', 'Consumer Staples', 'Energy', 'Utilities', 'Real Estate', 'Materials', 'General Market', 'None']
  dataset_name: danidanou/Bloomberg_Financial_News
  dataset_dir: ../data/
  rss_feeds: ['https://feeds.bloomberg.com/news/news.rss', 'https://feeds.bloomberg.com/markets/news.rss', 'https://feeds.bloomberg.com/business/news.rss', 'https://feeds.bloomberg.com/technology/news.rss', 'https://feeds.bloomberg.com/politics/news.rss', 'https://feeds.bloomberg.com/wealth/news.rss', 'https://feeds.bloomberg.com/economics/news.rss', 'https://feeds.bloomberg.com/green/news.rss', 'https://feeds.bloomberg.com/pursuits/news.rss', 'https://feeds.bloomberg.com/opinion/news.rss', 'https://feeds.bloomberg.com/finance/news.rss', 'http

bloomberg_financial_data.parquet.gzip:   0%|          | 0.00/482M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/446762 [00:00<?, ? examples/s]


--- Download Successful! ---


Map:   0%|          | 0/446762 [00:00<?, ? examples/s]

Training dataset processed.

--- Starting validation of 446762 entries ---
--- Validation Complete! ---
Training dataset validated.
Saving processed dataset to local cache at '../data/danidanou_Bloomberg_Financial_News_train.parquet'...
Total number of rows: 446762
Loading pipeline completed.


In [10]:
train_ds[0]

BloombergNewsEntry(Headline='Ivory Coast Keeps Cocoa Export Tax Below 22%, Document Shows', Date='2011-10-06', Link='http://www.bloomberg.com/news/2011-10-06/ivory-coast-keeps-cocoa-export-tax-below-22-document-shows.html', Article='Export taxes on cocoa beans from Ivory Coast , the world’s biggest producer of the chocolate ingredient, won’t exceed 22 percent of the international price this season, meeting a commitment to the International Monetary Fund , according to a finance ministry document. In the 2008-9 season taxes averaged 25.3 percent of international prices, the IMF said in a document posted on its website in November last year. While the country met the commitment in the season just ended, it had a change in government earlier this year. The rate meets a demand by the International Monetary Fund and the World Bank to reform the Ivorian cocoa and coffee industries in order to comply with the terms of its Heavily Indebted Poor Countries’ debt-relief program. Last year, the fi

In [14]:
from transducer import NewsTransducer
import nest_asyncio
nest_asyncio.apply()

processor = NewsTransducer(config)
results = await processor.process_news_entries_async(train_ds[:50], config.prompt_instructions, save_path_prefix="processed_news")

Processing news: 100%|██████████| 50/50 [00:30<00:00,  1.64entry/s]


In [16]:
import json
dict_data = [obj.model_dump() for obj in results]

# Pretty print using json.dumps
print(json.dumps(dict_data, indent=4))

[
    {
        "Headline": "Hungary Focusing on Debt, Deficit Reductions in 2012, Simor Says",
        "Date": "2011-10-06",
        "Link": "http://www.bloomberg.com/news/2011-10-06/hungary-focusing-on-debt-deficit-reductions-in-2012-simor-says.html",
        "Article": "Hungary\u2019s government is embarking on a tightening of budget policy in 2012 with a focus on reducing the debt and deficit levels, central bank President Andras Simor said. \u201cThe government is acutely aware that debt needs to be brought down and budget policy needs to be extremely disciplined,\u201d Simor said in a speech in London today. To contact the reporter on this story: Agnes Lovasz in London at  alovasz@bloomberg.net  To contact the editor responsible for this story: Zoltan Simon at  zsimon@bloomberg.net",
        "Industry": "General Market",
        "KeyPoints": "- Hungary's government plans tightening of budget policy in 2012.\n- Focus on reducing national debt and budget deficit.\n- Central Bank Pr